# sq_ai – Colab training notebook

Trains LightGBM on NSE500 daily bars (yfinance) and exports `lgb_trading_model.pkl` + `feature_names.txt` to your Google Drive.

**Hardware:** Colab free T4 is plenty for LightGBM. The whole notebook runs in <5 min.

**Steps:**
1. `Runtime ▸ Change runtime type ▸ T4 GPU` (optional – CPU is enough).
2. Run all cells top-to-bottom.
3. Download `lgb_trading_model.pkl` and `feature_names.txt` from Drive into `~/0to100/models/`.

In [ ]:
!pip -q install lightgbm yfinance joblib pandas numpy scikit-learn

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
OUT = '/content/drive/MyDrive/sq_ai_models'
import os; os.makedirs(OUT, exist_ok=True)

In [ ]:
import numpy as np
import pandas as pd
import yfinance as yf
import joblib
import lightgbm as lgb
from sklearn.metrics import roc_auc_score

# Edit this list to match your investable universe
SYMBOLS = ['RELIANCE.NS','TCS.NS','HDFCBANK.NS','INFY.NS','ICICIBANK.NS',
           'HINDUNILVR.NS','ITC.NS','SBIN.NS','BHARTIARTL.NS','KOTAKBANK.NS',
           'LT.NS','AXISBANK.NS','ASIANPAINT.NS','MARUTI.NS','TITAN.NS',
           'BAJFINANCE.NS','HCLTECH.NS','WIPRO.NS','ULTRACEMCO.NS','SUNPHARMA.NS']

FEATURES = ['sma_20','sma_50','volatility_20','momentum_5d',
            'volume_trend','rsi','atr','regime']

def engineer(df):
    out = df.copy()
    out.columns = [c[0].lower() if isinstance(c,tuple) else c.lower() for c in out.columns]
    out['sma_20'] = out['close'].rolling(20).mean()
    out['sma_50'] = out['close'].rolling(50).mean()
    ret = out['close'].pct_change()
    out['volatility_20'] = ret.rolling(20).std()*np.sqrt(252)
    out['momentum_5d'] = out['close'].pct_change(5)
    out['volume_trend'] = out['volume'] / out['volume'].rolling(20).mean()
    delta = out['close'].diff()
    gain = delta.clip(lower=0).rolling(14).mean()
    loss = -delta.clip(upper=0).rolling(14).mean()
    rs = gain / loss.replace(0, np.nan)
    out['rsi'] = (100 - 100/(1+rs)).fillna(50)
    tr = pd.concat([(out['high']-out['low']),
                    (out['high']-out['close'].shift()).abs(),
                    (out['low']-out['close'].shift()).abs()], axis=1).max(1)
    out['atr'] = tr.rolling(14).mean()
    out['regime'] = np.where(out['sma_20'] > out['sma_50']*1.005, 2,
                              np.where(out['sma_20'] < out['sma_50']*0.995, 0, 1))
    out['target'] = (out['close'].shift(-5)/out['close'] - 1 > 0.01).astype(int)
    return out.dropna()

frames = []
for s in SYMBOLS:
    raw = yf.download(s, start='2018-01-01', end='2024-01-01', progress=False, auto_adjust=False)
    if len(raw) < 200: continue
    f = engineer(raw); f['symbol'] = s
    frames.append(f)
all_df = pd.concat(frames).sort_index()
print('rows:', len(all_df))

In [ ]:
split = '2022-01-01'
train = all_df[all_df.index < split]
test  = all_df[all_df.index >= split]
X_tr, y_tr = train[FEATURES], train['target']
X_te, y_te = test[FEATURES],  test['target']
clf = lgb.LGBMClassifier(n_estimators=600, learning_rate=0.04, max_depth=6,
                         num_leaves=63, n_jobs=-1, verbose=-1)
clf.fit(X_tr, y_tr, eval_set=[(X_te, y_te)])
p_te = clf.predict_proba(X_te)[:,1]
print('AUC OOS:', roc_auc_score(y_te, p_te))

In [ ]:
joblib.dump(clf, f'{OUT}/lgb_trading_model.pkl')
with open(f'{OUT}/feature_names.txt','w') as f:
    f.write('\n'.join(FEATURES))
print('Saved →', OUT)
print('Download these two files into ~/0to100/models/ on your laptop.')

## Walk-forward 2022 (out-of-sample)
Re-uses the identical feature engineering as the live pipeline so backtest = live.

In [ ]:
import json
# !curl -O https://raw.githubusercontent.com/.../sq_ai/train/walk_forward.py    # if hosted
from sq_ai.train.walk_forward import walk_forward    # adapt path as needed
print(json.dumps(walk_forward('RELIANCE.NS'), indent=2))